# Montreal electoral districts: real boundaries, real attributes

This notebook starts with real administrative-style polygons bundled as GeoJSON. It avoids synthetic boundaries and demonstrates a choropleth, hover labels, and point overlays.

Data: `montreal_districts.geojson`, `election.csv`, and `carshare.csv` from the Plotly example datasets packaged with Plotly. The district polygons represent Montreal electoral districts.

Video prompt: watch a short geospatial visualization overview before running the map. <iframe width="560" height="315" src="https://www.youtube.com/embed/hgL_O1iVwQ4" title="Geospatial data visualization" frameborder="0" allowfullscreen></iframe>

**Reflection questions:** How does polygon size influence interpretation? What extra metadata would you need before using district-level political data for public-health planning? What can a point layer reveal that a choropleth cannot?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
districts = load_json('montreal_districts.geojson')
election = load_csv('election.csv')
carshare = load_csv('carshare.csv')

# Merge election attributes into the GeoJSON feature properties.
e = election.set_index('district').to_dict(orient='index')
for feature in districts['features']:
    d = feature['properties']['district']
    feature['properties'].update(e.get(d, {}))

m = folium.Map(location=[45.52, -73.60], zoom_start=11, tiles='CartoDB positron')
folium.Choropleth(
    geo_data=districts,
    data=election,
    columns=['district','total'],
    key_on='feature.properties.district',
    fill_color='YlOrRd',
    fill_opacity=0.72,
    line_opacity=0.35,
    legend_name='Votes cast by district'
).add_to(m)
folium.GeoJson(
    districts,
    name='District labels',
    tooltip=folium.GeoJsonTooltip(fields=['district','winner','total'], aliases=['District','Winner','Votes'])
).add_to(m)
for _, row in carshare.head(120).iterrows():
    folium.CircleMarker([row.centroid_lat, row.centroid_lon], radius=3, fill=True, popup=f"Car-share hours: {row.car_hours:.0f}").add_to(m)
add_standard_controls(m)
m